# DepMap Chronos dependency analysis in TNBC cell-line models

## Overview

This notebook analyses the association between endogenous *MTUS1* expression and CRISPR--Cas9 gene dependency scores in TNBC cell-line models. The analysis uses DepMap Chronos gene-effect scores to identify genes whose loss has a stronger fitness effect in models with lower *MTUS1* expression, and then tests whether these genes are enriched in cancer-relevant pathways.

The notebook implements the analysis used for the article. Cell-line metadata are merged with mutation annotations using a left join (`how='left'`) to retain models with a DepMap identifier even when a COSMIC identifier is unavailable. Genes are filtered before correlation testing to retain genes with evidence of essentiality (`min_effect < -1`), reducing the number of tests and the multiple-testing burden. Protein-level data are included only as exploratory supporting information and should be interpreted cautiously.


In [ ]:
from scipy.stats import pearsonr
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


# 0. Data sources

Raw input files are not committed to the repository. To reproduce the analysis, download the required DepMap, PRISM/GDSC, and protein data files from the official portals and place them under `data/raw/Cell-lines/`, as described in the repository README.

Main resources used in this notebook include:

- DepMap portal: model metadata, CRISPR Chronos gene-effect scores, and RNA expression data.
- Genomics of Drug Sensitivity in Cancer (GDSC): genetic feature annotations.
- Protein-level datasets used for exploratory supporting analyses.


# 1. Load cell-line data

## A. Mutation (cell line name/COSMIC_ID/GDSC Desc1/GDSC Desc2/TCGA Desc)


In the file `PANCANCER_Genetic_features.csv`, downloaded from  [Genomics of Drug Sensitivity in Cancer (GDSC)](https://www.cancerrxgene.org/downloads),  
we have access to the following information for each cell line:

- **Cell Line Name**
- **COSMIC_ID**
- **GDSC Desc1**
- **GDSC Desc2**
- **TCGA Desc**
- and the list of **mutations** identified in each sample.

In [ ]:
# Load the genetic features dataset from a CSV file
Genetic_features = pd.read_csv("../data/raw/Cell-lines/PANCANCER_Genetic_features_2025_04_07.csv")

# Rename the column 'COSMIC ID' to 'COSMIC_ID' for easier access and consistency
Genetic_features.rename(columns={'COSMIC ID': 'COSMIC_ID'}, inplace=True)

# Show to explore the dataset
# - the tumor type (TCGA Desc) is 'BRCA' (breast cancer)
# - and the genetic feature is 'KRAS_mut' (mutation in KRAS gene): only 2 cell lines have this mutation
Genetic_features[(Genetic_features["TCGA Desc"] == 'BRCA') & 
                 (Genetic_features['Genetic Feature'] == 'KRAS_mut')]


In [ ]:
# Create a new column with the mutated genes
def get_mutated_genes(row):
    mutated_genes = []

    # Check if the feature is marked as mutated
    if row['IS Mutated'] == 1:
        
        # Check if the gene name contains "_mut" in the 'Genetic Feature' column
        if '_mut' in row['Genetic Feature']:
            gene_name = row['Genetic Feature'].replace('_mut', '')
            mutated_genes.append(gene_name)
        
        # Optionally, check for genes listed in the 'Genes in Segment' column
        # Uncomment the following block if you want to include these genes as well
        # if pd.notna(row['Genes in Segment']):
        #     genes_list = row['Genes in Segment'].split(',')
        #     mutated_genes.extend(genes_list)
    
    # Return a list of unique gene names (remove duplicates)
    return list(set(mutated_genes))

# Apply the function to each row of the DataFrame to create the 'Mutated Genes' column
Genetic_features['Mutated Genes'] = Genetic_features.apply(get_mutated_genes, axis=1)

# Aggregate genetic features by cell line, keeping the first descriptions and merging unique mutated genes per group
grouped_Genetic_features = Genetic_features.groupby(['Cell Line Name', 'COSMIC_ID']).agg({
    'GDSC Desc1': 'first',
    'GDSC Desc2': 'first',
    'TCGA Desc': 'first',
    'Mutated Genes': lambda x: list(set([gene for sublist in x for gene in sublist]))
}).reset_index()
grouped_Genetic_features

In [ ]:
TCGA_type = 'BRCA'  # Example TCGA type, can be changed as needed
# Filter the grouped DataFrame for a specific TCGA type (e.g., 'BRCA')
print("number of cell lines:",len(grouped_Genetic_features[(grouped_Genetic_features["TCGA Desc"] == TCGA_type)]))
grouped_Genetic_features[(grouped_Genetic_features["TCGA Desc"] == TCGA_type)]

In [ ]:
# List of cell lines with specific mutated genes
list_mut = ['KRAS','BRAF']#,'HRAS','BRAF']
grouped_Genetic_features[(grouped_Genetic_features["TCGA Desc"] == 'BRCA') & (grouped_Genetic_features["Mutated Genes"].apply(lambda x: any(gene in x for gene in list_mut)))]

## B. Metadata (DepMap_ID / COSMICID)

In the file `Model.csv`, downloaded from the  [DepMap portal](https://depmap.org/portal/data_page/?tab=allData),  
we have access to metadata describing all cancer models (cell lines) that are referenced by datasets included in the DepMap portal.  
This includes key identifiers such as **DepMap_ID** and **COSMICID**.

In [ ]:
meta = pd.read_csv("../data/raw/Cell-lines/Model.csv") 
meta.rename(columns={'ModelID': 'DepMap_ID', 'COSMICID':'COSMIC_ID' }, inplace=True)
meta[meta["COSMIC_ID"] == 906863]

In [ ]:
meta[meta['OncotreeLineage'].str.contains('Breast', na=False)]

In [ ]:
meta[meta["DepMap_ID"] == 'ACH-000721']

In [ ]:
meta_grouped_Genetic_features = meta.merge(grouped_Genetic_features, on='COSMIC_ID', how='left')
meta_grouped_Genetic_features

In [ ]:
TCGA_type = 'BRCA'  # Example TCGA type, can be changed as needed
print("number of cell lines:",len(meta_grouped_Genetic_features[(meta_grouped_Genetic_features["TCGA Desc"] == TCGA_type)]))

In [ ]:
print("number of cell lines:",len(meta_grouped_Genetic_features[meta_grouped_Genetic_features['OncotreeLineage'].str.contains('Breast', na=False)]))

## C. Expression (Gene/Protein)

### 1. Gene Expression (DepMap 24Q4)

We use the file `OmicsExpressionProteinCodingGenesTPMLogp1BatchCorrected.csv` from the  [DepMap portal](https://depmap.org/portal/download/), which contains batch-corrected log-transformed TPM values  for protein-coding genes across cancer cell lines.

Steps:
- Set `DepMap_ID` as a column for merging with other datasets
- Clean column names by removing gene annotations in parentheses (e.g., gene IDs or descriptions)


1673 cell lines/ 19139 genes

In [ ]:
# Load DepMap 24Q4 expression data (log-transformed TPM, batch corrected)
# Set 'DepMap_ID' as a column instead of index
# Clean column names by removing anything in parentheses and trimming whitespace
df = pd.read_csv("../data/raw/Cell-lines/OmicsExpressionProteinCodingGenesTPMLogp1BatchCorrected.csv", index_col=0) #DepMap Public 24Q4
df.index.name = 'DepMap_ID'
df = df.reset_index()
df.columns = df.columns.str.extract(r'^([^\(]+)', expand=False).str.strip()
df.rename(columns={'COSMICID': 'COSMIC_ID'}, inplace=True)
print(df.shape)
df.head(5)
#df = df[['DepMap_ID','MTUS1','MYC','MAPT','MAPK3','MAPK1','MAP2K1','MAP2K2','JAK1','JAK2','JAK3','AKT1','AKT2','AKT3','PIK3CA','KRAS', 'BRAF', 'NRAS','RAF1','HRAS','KIF2A','MAST4',]]

### 2. Proteomics (Mass Spectrometry, CCLE / DepMap)

We replace the older CCLE RPPA (2018) data with a more comprehensive mass‑spectrometry proteomics dataset provided by DepMap.  
The file `protein_quant_current_normalized.csv` (e.g. "proteomic_20Q2") includes normalized, quantitative protein expression across ~375 CCLE cell lines and covers thousands of protein targets—far beyond the limited panel in the original RPPA data.

This extended dataset allows deeper analysis of protein-level changes and better integration with transcriptomics, drug sensitivity, and mutation data.

426 cell lines, 12 755 proteins

In [ ]:
df_p2 = pd.read_csv("../data/raw/Cell-lines/protein_quant_current_normalized_2020.csv")
df_p2.columns, df_p2.shape

In [ ]:
# MTUS1 quantification across 333 cell lines.
df_p_MTUS1 = df_p2[df_p2['Gene_Symbol'] == 'MTUS1'].T.dropna().iloc[48:]
df_p_MTUS1.columns = ['MTUS1']
df_p_MTUS1 = df_p_MTUS1.reset_index()
df_p_MTUS1.rename(columns={'index': 'Name'}, inplace=True)
# Split the 'Name' column into cell line name and tissue type
df_p_MTUS1[['CELL_LINE_NAME', 'TISSUE_TYPE']] = df_p_MTUS1['Name'].str.split('_', n=2, expand=True)[[0, 1]]

# Display the first rows to check the result
df_p_MTUS1[['Name', 'CELL_LINE_NAME', 'TISSUE_TYPE','MTUS1']].head()

In [ ]:
# chnager CELL_LINE_NAME en CellLineName
df_p_MTUS1.rename(columns={'CELL_LINE_NAME': 'StrippedCellLineName'}, inplace=True)
# Merge with metadata to add DepMap_ID
df_p_MTUS1 = df_p_MTUS1.merge(meta[['DepMap_ID', 'StrippedCellLineName']], on='StrippedCellLineName', how='left')
df_p_MTUS1

## D. Select Cell-ligns

In [ ]:
def load_data(type_data, dict_filters, list_mutation,
               drop_mutation=False, keep_only_mutation=False, expr_keep=[]):
    """
    Load and filter omics data (RNA/protein) with filtering based on metadata and mutation status.

    Args:
        type_data (str): 'TPMLogp1' or 'protein_quant'.
        dict_filters (dict): Dictionary of filtering conditions (e.g., {'TCGA_DESC': lambda col: col == 'BRCA'}).
        list_mutation (list): List of genes to use for mutation filtering.
        drop_mutation (bool): If True, exclude mutated samples.
        keep_only_mutation (bool): If True, keep only mutated samples.
        expr_keep (list): Genes to retain (only used for TPMLogp1).

    Returns:
        pd.DataFrame: Filtered expression data.
    """

    # Step 1: Merge meta and grouped_Genetic_features once
    
    # meta....
    meta = pd.read_csv("../data/raw/Cell-lines/Model.csv") 
    meta.rename(columns={'ModelID': 'DepMap_ID', 'COSMICID':'COSMIC_ID' }, inplace=True)

    meta_combined = pd.merge(meta, grouped_Genetic_features, on='COSMIC_ID', how='left') # how='inner' was used in the thesis chapter; use how='left' here to retain cell lines with a DepMap ID but no COSMIC ID.

    # Step 2: Load expression data
    if type_data == 'TPMLogp1':
        df_expr = pd.read_csv("../data/raw/Cell-lines/OmicsExpressionProteinCodingGenesTPMLogp1BatchCorrected.csv", index_col=0)
        df_expr.index.name = 'DepMap_ID'
        df_expr = df_expr.reset_index()
        df_expr.columns = df_expr.columns.str.extract(r'^([^\(]+)', expand=False).str.strip()
        if expr_keep:
            df_expr = df_expr[['DepMap_ID'] + expr_keep]

    elif type_data == 'protein_quant':
        df_p2 = pd.read_csv("../data/raw/Cell-lines/protein_quant_current_normalized_2020.csv")
        df_expr = df_p2[df_p2['Gene_Symbol'] == 'MTUS1'].T.dropna().iloc[48:]
        df_expr.columns = ['MTUS1']
        df_expr['MTUS1'] = pd.to_numeric(df_expr['MTUS1'], errors='coerce')
        df_expr = df_expr.reset_index()
        df_expr.rename(columns={'index': 'Name'}, inplace=True)
        df_expr[['CELL_LINE_NAME', 'TISSUE_TYPE']] = df_expr['Name'].str.split('_', n=2, expand=True)[[0, 1]]
        df_expr.rename(columns={'CELL_LINE_NAME': 'StrippedCellLineName'}, inplace=True)
        df_expr = df_expr.merge(meta[['DepMap_ID', 'StrippedCellLineName']], on='StrippedCellLineName', how='left')
        df_expr = df_expr[['DepMap_ID', 'MTUS1']]

    # Step 3: Merge expression data with full metadata (including mutations)
    df_full = df_expr.merge(meta_combined, on=['DepMap_ID'], how='inner') 

    # Step 4: Apply all filters from dict_filters
    for col, condition in dict_filters.items():
        if col in df_full.columns:
            df_full = df_full[condition(df_full[col])]

    # Step 5: Apply mutation filters
    mutation_mask = df_full["Mutated Genes"].apply(
        lambda x: any(g in x for g in list_mutation) if isinstance(x, list) else False
    )

    if drop_mutation:
        df_full = df_full[~mutation_mask]

    if keep_only_mutation:
        df_full = df_full[mutation_mask]

    return df_full


In [ ]:
# filter data type
type_data = 'TPMLogp1'#'TPMLogp1'#'protein_quant'#'TPMLogp1' #'protein_quant'

# Define filtering conditions for the dataset
conditions = {
    #'TCGA Desc': lambda col: col.str.contains('PAAD', na=False),
    #'OncotreeSubtype': lambda col: col.str.contains('Breast', na=False),
    #'OncotreeCode':lambda col: col.str.contains('COAD', na=False),              # Similar to merged_df = merged_df[merged_df['TCGA_DESC'] == 'BLCA'], but incorrect for LGG and BRCA.
    'ModelSubtypeFeatures': lambda col: col.str.contains('TNBC', na=False),
    #'PrimaryOrMetastasis': lambda col: col.str.contains('Primary', na=False),   
}

# Load and filter mutation data
#list_mutation = ['KRAS', 'BRAF', 'NRAS','RAF1','HRAS']#,'PIK3CA','KRAS', 'BRAF', 'NRAS','RAF1','HRAS'
list_mutation = ['PIK3CA','PTEN']
drop_mut = False
keep_only_mut = False

# Define genes to keep in the expression data
expr_keep=['BAG2','DAD1','HMGB2','RPS16','PSMB5','DDX10','MTUS1','FOXO3', 'CDX2','MYC','MAPT','MAPK3','MAPK1','MAP2K1','MAP2K2','JAK1','JAK2','JAK3','AKT1','AKT2','AKT3','PIK3CA','KRAS', 'BRAF', 'NRAS','RAF1','HRAS','KIF2A','MAST4','WEE1','HSP90AB1','HSP90AA1','BRD4','CHEK1','CHEK2','ATM','ATR','AURKB','AURKA','SMC1A','CDC45','PLK1'] #,'TP53','CDKN2A','RB1','MTUS1' #,'MTUS1','ATIP3a','ATIP1','ATIP4','ATIP3b'

merged_df = load_data(type_data,conditions,list_mutation, drop_mutation = drop_mut, keep_only_mutation = keep_only_mut,expr_keep = expr_keep)
print("nombre de cell lines", merged_df['DepMap_ID'].drop_duplicates().shape[0])
# Display the first few rows of the merged DataFrame
merged_df

With RNA-seq: 30 cell lines; with protein quantification: 15 cell lines.

# 2. Vulnerability data 

Data downloaded from the DepMap portal.

# CHRONOS (derived from CRISPR knock-out)
https://genomebiology.biomedcentral.com/articles/10.1186/s13059-021-02540-7


The Chronos score, derived from CRISPR knockout screens, measures the effect of gene loss on cell proliferation:

A **negative Chronos score (< 0)** indicates that the gene is **essential**: its inactivation decreases cell growth.

A Chronos score close to 0 or **positive (>= 0)** indicates that the gene is **non-essential**.

1100 cell lines et 18444 genes

## A. Data loading

In [ ]:
import pandas as pd

# Load the raw file.
effect_df = pd.read_csv("../data/raw/Cell-lines/CRISPRGeneEffect.csv")

# Rename the first column.
effect_df.rename(columns={effect_df.columns[0]: "DepMap_ID"}, inplace=True)

# Clean gene names in column headers by keeping only the gene symbol.
new_columns = [col.split(" ")[0] if col != "DepMap_ID" else col for col in effect_df.columns]
effect_df.columns = new_columns

# Check.
print("Dimensions :", effect_df.shape)
print(effect_df.iloc[:5, :5])


In [ ]:
# Merge effect_df with merged_df to keep only cell lines present in merged_df.
effect_df = effect_df.merge(merged_df[['DepMap_ID']], on='DepMap_ID', how='inner')
print("nombre de cell lines", effect_df['DepMap_ID'].drop_duplicates().shape[0])
# Display the first few rows of the effect DataFrame
effect_df

In [ ]:
# Keep only genes whose values are all below -1.
# Select only numeric columns (all genes).
num_df = effect_df.drop(columns=["DepMap_ID"])

# Keep only columns where the maximum is < -1.
essential_genes = num_df.columns[(num_df.max(axis=0) < -1)]
print(f"Number of essential genes (max < -1): {len(essential_genes)}")

In [ ]:
#for essential_gene in essential_genes:
#    print(essential_gene)

In [ ]:
# Keep genes with at least one value below -1; this reduces the set from all genes to 1,854 genes (10%).
# Select only numeric columns (all genes).
num_df = effect_df.drop(columns=["DepMap_ID"])

# Keep only columns where the minimum is < -1.
genes_to_keep = num_df.columns[(num_df.min(axis=0) < -1)]

# Keep DepMap_ID and filtered genes.
filtered_df = effect_df[["DepMap_ID"] + genes_to_keep.tolist()]
filtered_df
filtered_df.shape

In [ ]:
# Test only a few genes: those with the largest AUC or at least one value below -1.

stats = filtered_df.std(numeric_only=True)
stats = stats.reset_index()
stats.columns = ["Gene", "sd"]
stats.sort_values('sd', ascending=False).head(20)
plt.hist(stats["sd"])#.sort_values('sd', ascending=False)

In [ ]:
stats[stats['Gene'] == 'KAT5'] #'HDAC3', 'MYC', KAT5

In [ ]:
# keep genes with sd > 75th percentile (the genes with the highest variability)

thr = stats["sd"].quantile(0.75)      
keep = stats[stats["sd"] > thr]['Gene'].unique()
print(thr)
filtered_df_filter = effect_df[['DepMap_ID']+list(keep)]
effect_df.shape,filtered_df_filter.shape
filtered_df_filter.shape

## B. Correlation between ATIP3 expression and gene Chronos score, compared with Chronos score sign

 | **Correlation between ATIP3 expression and gene Chronos score** | **Gene Chronos score** | **When ATIP3 is low** | **Biological interpretation** |
| -------------------------------- | --------------------------- | ---------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------ |
| **Positive** | **Negative** (essential) | The gene becomes **more essential** | Functional dependency: the gene supports survival and is a possible therapeutic target in an ATIP3-low context. |
| **Positive** | **Positive** (non-essential) | The gene remains non-essential | Limited effect: the gene may be weakly involved, not required, or co-expressed without functional impact. |


In [ ]:
from scipy.stats import spearmanr
def correlate_gene_effect_with_expression(effect_df, expression_df, gene='WEE1', expr_col='MTUS1'):
    """
    Correlate dependency on one gene (CRISPRGeneEffect) with expression of another gene (e.g. MTUS1).
    """
    # S'assurer que DepMap_ID est bien une colonne
    if effect_df.index.name == 'DepMap_ID' or 'DepMap_ID' not in effect_df.columns:
        effect_df.index.name = 'DepMap_ID'
        effect_df = effect_df.reset_index()
    if expression_df.index.name == 'DepMap_ID' or 'DepMap_ID' not in expression_df.columns:
        expression_df.index.name = 'DepMap_ID'
        expression_df = expression_df.reset_index()
        
    # Uniformiser le type de DepMap_ID
    effect_df['DepMap_ID'] = effect_df['DepMap_ID'].astype(str)
    expression_df['DepMap_ID'] = expression_df['DepMap_ID'].astype(str)

    # Merge and compute correlation.
    sub = expression_df[['DepMap_ID', expr_col]].merge(effect_df[['DepMap_ID', gene]], on='DepMap_ID')
    sub = sub.dropna()
    corr, pval = pearsonr(sub[expr_col], sub[gene])
    #corr, pval = spearmanr(sub[expr_col], sub[gene])
    return corr, pval

gene = 'MYC'
expr_col='MTUS1'
corr, pval = correlate_gene_effect_with_expression(effect_df, merged_df, gene=gene , expr_col=expr_col)
print(f"the correlation between the Chronos score of {gene} and the expression of {expr_col} is {corr}, with p-value: {pval}")


In [ ]:
from statsmodels.stats.multitest import multipletests
from scipy.stats import spearmanr

def correlate_selected_genes_with_expression(effect_df, expression_df, expr_col='MTUS1', genes=[]):
    results = []
    effect_df['DepMap_ID'] = effect_df['DepMap_ID'].astype(str)
    expression_df['DepMap_ID'] = expression_df['DepMap_ID'].astype(str)
    if genes == []:
        genes = effect_df.columns[1:].to_list()
        # on remove MTUS1
        if expr_col in genes:
            genes.remove(expr_col)
    
    for gene in genes:
        if gene not in effect_df.columns:
            continue
        merged = pd.merge(
            expression_df[['DepMap_ID', expr_col]],
            effect_df[['DepMap_ID', gene]],
            on='DepMap_ID'
        ).dropna()
        if len(merged) >= 3:
            corr, pval = pearsonr(merged[expr_col], merged[gene])
            corr2, pval2  = spearmanr(merged[expr_col], merged[gene])
            results.append({'Gene': gene, 'Pearson Correlation': corr, 'p_value': pval, 'Spearman Correlation': corr2, 'Spearman p_value': pval2, 'n': len(merged), 'min_score': merged[gene].min(), 'max_score': merged[gene].max()})
    
    res_df = pd.DataFrame(results)
    # adjusted p-value
    # Apply Benjamini-Hochberg correction
    if not res_df.empty:
        _, fdr_corrected_pvals, _, _ = multipletests(res_df['p_value'], method='fdr_bh')
        res_df['p_value_FDR'] = fdr_corrected_pvals
        _, fdr_corrected_pvals, _, _ = multipletests(res_df['Spearman p_value'], method='fdr_bh')
        res_df['Spearman p_value_FDR'] = fdr_corrected_pvals

    return res_df.sort_values('p_value')

In [ ]:
# Test only a few genes: those with the largest AUC or at least one value below -1.

stats = effect_df.std(numeric_only=True)
stats = stats.reset_index()
stats.columns = ["Gene", "sd"]
plt.hist(stats["sd"])#.sort_values('sd', ascending=False)

In [ ]:
# Extract the table for the 21/24 cell lines, compute standard deviation on this subset, and filter on this subset; this necessarily yields fewer genes than filtering the complete table.

thr = stats["sd"].quantile(0.75)      
keep = stats[stats["sd"] > thr]['Gene'].unique()
print(thr)
effect_df_filtered = effect_df[['DepMap_ID']+list(keep)] # This reduces the list from all genes to approximately 4,000 genes.
# It is also possible to keep only columns whose minimum is below -1.
effect_df_filtered_2 = effect_df_filtered[effect_df_filtered[keep].min(axis=1) < -1]
print(effect_df_filtered_2.shape, effect_df_filtered.shape)
effect_df_filtered_2

In [ ]:
# test only few genes 
genes_to_test_MYC_targets_V1 = ['ABCE1','ACP1','AIMP2','AP3S1','APEX1','BUB3','C1QBP','CAD','CANX','CBX3','CCNA2','CCT2','CCT3','CCT4','CCT5','CCT7','CDC20','CDC45','CDK2','CDK4','CLNS1A','CNBP','COPS5','COX5A','CSTF2','CTPS1','CUL1','CYC1','DDX18','DDX21','DEK','DHX15','DUT','EEF1B2','EIF1AX','EIF2S1','EIF2S2','EIF3B','EIF3D','EIF3J','EIF4A1','EIF4E','EIF4G2','EIF4H','EPRS1','ERH','ETF1','EXOSC7','FAM120A','FBL','G3BP1','GLO1','RACK1','GNL3','GOT2','GSPT1','H2AZ1','HDAC2','HDDC2','HDGF','HNRNPA1','HNRNPA2B1','HNRNPA3','HNRNPC','HNRNPD','HNRNPR','HNRNPU','HPRT1','HSP90AB1','HSPD1','HSPE1','IARS1','IFRD1','ILF2','IMPDH2','KARS1','KPNA2','KPNB1','LDHA','LSM2','LSM7','MAD2L1','MCM2','MCM4','MCM5','MCM6','MCM7','MRPL23','MRPL9','MRPS18B','MYC','NAP1L1','NCBP1','NCBP2','NDUFAB1','NHP2','NME1','NOLC1','NOP16','NOP56','NPM1','ODC1','ORC2','PA2G4','PABPC1','PABPC4','PCBP1','PCNA','PGK1','PHB1','PHB2','POLD2','POLE3','PPIA','PPM1G','PRDX3','PRDX4','PRPF31','PRPS2','PSMA1','PSMA2','PSMA4','PSMA6','PSMA7','PSMB2','PSMB3','PSMC4','PSMC6','PSMD1','PSMD14','PSMD3','PSMD7','PSMD8','PTGES3','PWP1','RAD23B','RAN','RANBP1','RFC4','RNPS1','RPL14','RPL18','RPL22','RPL34','RPL6','RPLP0','RPS10','RPS2','RPS3','RPS5','RPS6','RRM1','RRP9','RSL1D1','RUVBL2','SERBP1','SET','SF3A1','SF3B3','SLC25A3','SMARCC1','SNRPA','SNRPA1','SNRPB2','SNRPD1','SNRPD2','SNRPD3','SNRPG','SRM','SRPK1','SRSF1','SRSF2','SRSF3','SRSF7','SSB','SSBP1','STARD7','SYNCRIP','TARDBP','TCP1','TFDP1','TOMM70','TRA2B','TRIM28','TUFM','TXNL4A','TYMS','U2AF1','UBA2','UBE2E1','UBE2L3','USP1','VBP1','VDAC1','VDAC3','XPO1','XPOT','XRCC6','YWHAE','YWHAQ']

genes_to_test_MYC_targets_V2 = ['AIMP2','BYSL','CBX3','CDK4','DCTPP1','DDX18','DUSP2','EXOSC5','FARSA','GNL3','GRWD1','HK2','HSPD1','HSPE1','IMP4','IPO4','LAS1L','MAP3K6','MCM4','MCM5','MPHOSPH10','MRTO4','MYBBP1A','MYC','NDUFAF4','NIP7','NOC4L','NOLC1','NOP16','NOP2','NOP56','NPM1','PA2G4','PES1','PHB1','PLK1','PLK4','PPAN','PPRC1','PRMT3','PUS1','RABEPK','RCL1','RRP12','RRP9','SLC19A1','SLC29A2','SORD','SRM','SUPV3L1','TBRG4','TCOF1','TFB2M','TMEM97','UNG','UTP20','WDR43','WDR74']

genes_to_test_UPR = ['ALDH18A1','ARFGAP1','ASNS','ATF3','ATF4','ATF6','ATP6V0D1','BAG3','BANF1','CALR','CCL2','CEBPB','CEBPG','CHAC1','CKS1B','CNOT2','CNOT4','CNOT6','CXXC1','DCP1A','DCP2','DCTN1','DDIT4','DDX10','DKC1','DNAJA4','DNAJB9','DNAJC3','EDC4','EDEM1','EEF2','EIF2AK3','EIF2S1','EIF4A1','EIF4A2','EIF4A3','EIF4E','EIF4EBP1','EIF4G1','ERN1','ERO1A','EXOC2','EXOSC1','EXOSC10','EXOSC2','EXOSC4','EXOSC5','EXOSC9','FKBP14','FUS','GEMIN4','GOSR2','H2AX','HERPUD1','HSP90B1','HSPA5','HSPA9','HYOU1','IARS1','IFIT1','IGFBP1','IMP3','KDELR3','KHSRP','KIF5B','LSM1','LSM4','MTHFD2','NFYA','NFYB','NHP2','NOLC1','NOP14','NOP56','NPM1','NABP1','PAIP1','PARN','PDIA5','PDIA6','POP4','PREB','PSAT1','RPS14','RRP9','SDAD1','SEC11A','SEC31A','SERP1','SHC1','MTREX','SLC1A4','SLC30A5','SLC7A5','SPCS1','SPCS3','SRPRA','SRPRB','SSR1','STC2','TARS1','TATDN2','TSPYL2','SKIC3','TUBB2A','VEGFA','WFS1','WIPI1','XBP1','XPOT','YIF1A','YWHAZ','ZBTB17']

genes_to_test_DNA_repair = ['AAAS','ADA','ADCY6','ADRM1','AK1','AK3','APRT','ARL6IP1','BCAM','BCAP31','BOLA2','BRF2','MPC2','CANT1','CCNO','CDA','CETN2','CLP1','CMPK2','NELFB','COX17','CSTF3','DAD1','DCTN4','DDB1','DDB2','GSDME','DGCR8','DGUOK','DUT','EDF1','EIF1B','AGO4','ELL','ERCC1','ERCC2','ERCC3','ERCC4','ERCC5','ERCC8','FEN1','GMPR2','GPX4','GTF2A2','GTF2B','GTF2F1','GTF2H1','GTF2H3','GTF2H5','GTF3C5','GUK1','HCLS1','HPRT1','IMPDH2','ITPA','LIG1','MPG','MRPL40','NCBP2','NFX1','NME1','NME3','NME4','NPR2','NT5C','NT5C3A','NUDT21','NUDT9','PCNA','PDE4B','PDE6G','PNP','POLA1','POLA2','POLB','POLD1','POLD3','POLD4','POLE4','POLH','POLL','POLR1C','POLR1D','POLR2A','POLR2C','POLR2D','POLR2E','POLR2F','POLR2G','POLR2H','POLR2I','POLR2J','POLR2K','POLR3C','POLR3GL','POM121','PRIM1','RAD51','RAD52','RAE1','RALA','RBX1','NELFE','REV3L','RFC2','RFC3','RFC4','RFC5','RNMT','RPA2','RPA3','RRM2B','SAC3D1','SDCBP','SEC61A1','SF3A3','SMAD5','SNAPC4','SNAPC5','SRSF6','SSRP1','STX3','SUPT4H1','SUPT5H','SURF1','TAF10','TAF12','TAF13','TAF1C','TAF6','TAF9','TARBP2','ELOA','NELFCD','ALYREF','TK2','TMED2','TP53','TSG101','TYMS','UMPS','UPF3B','USP11','VPS28','VPS37B','VPS37D','XPC','ZNF707','POLR1H','ZWINT']

#genes selectionated with MTUS1 expression (TPMlogp1) in 16 TNBC cell lines
genes_to_test_MTUS1_expression = ['DDX10','PSMB5', 'DYNLL1', 'MYC','GTF2H4','NOB1','VARS1','WDR46',
 'ACTL6A','RPS16','EIF3I','EIF2B5','DMAP1','DCTN4','MSTO1','BAP1','MLST8','EP400',
 'GGPS1','RPL12','TBL1XR1','PPAN','NOP9','PWP2','HDAC3','MASTL','RPL32','SRFBP1', 'FARSB','SAP18',
 'DDX24','RIOK1','FARSA','FGFR1',
 'NOL9', 'WARS1','OSGEP','IMP3', 'KRTAP4-2', 'TBL3','NOL6','TCP1','WDR18', 'RPL14','ABT1',
 'DDX49', 'CDC23', 'SERBP1',
 'EIF3F', 'SPCS2', 'THG1L', 'EEF2','RPS3A', 'DENR', 'TSR2']

#genes selectionated with MTUS1 protein expression in 14 TNBC cell lines
genes_to_test_MTUS1_protein_quant = ['METAP2',
 'CAD', 'ELP4', 'RBM42', 'GPATCH1',
 'NOL7', 'NUMA1', 'LAMTOR3',
 'NOB1', 'DDX24', 'APEX2', 'ABCF1',
 'FNTA', 'EIF2B2', 'COPB2',
 'UBIAD1', 'XRCC1', 'HNRNPA1',
 'RPL12', 'SAP30BP',
 'UMPS', 'ELMO2',
 'NOC3L', 'DNMT1', 'DAD1', 'LIG1',
 'TELO2','PCNA', 'POP4',
 'ILF2','TTI2','RNMT',
 'ANAPC4', 'DIMT1', 'USP36',
 'CIP2A', 'ATP6V1F','DHFR','DDOST', 'HTATSF1', 'PDRG1',
 'WDR48', 'GGPS1', 'TUBD1',
 'VPS41', 'LAMTOR4', 'DBR1',
 'POP1', 'SERBP1', 'WDR83', 'PDS5A',
 'AATF', 'COPE', 'FARSB', 'TIMM10', 'SEC61G', 'NAT10',
 'UTP3', 'NMD3','SRRM2', 'ABHD11','GMNN']

genes_to_test = ['BAG2', 'HSPA8', 'IRE1', 'USP49','ATF6','PERK','GRP78','SKP1']

selected_correlations = correlate_selected_genes_with_expression(filtered_df, merged_df, expr_col='MTUS1', genes=genes_to_test)

display(selected_correlations)

In [ ]:
print(effect_df.shape) # all_genes
print(filtered_df.shape) # genes with Chronos_min < -1
print(effect_df_filtered.shape) # genes with sd > 0.75
print(merged_df.shape) # Chronos score

In [ ]:
# try for all genes 
selected_correlations_all = correlate_selected_genes_with_expression(effect_df, merged_df, expr_col='MTUS1', genes=[])
print(selected_correlations_all.shape)
display(selected_correlations_all[(selected_correlations_all['p_value']< 0.05) & (selected_correlations_all['Pearson Correlation'] > 0) &  (selected_correlations_all['min_score'] < -1) ].sort_values('Pearson Correlation', ascending=False))

In [ ]:
selected_correlations_all[selected_correlations_all['Gene'] == 'BAG2']

In [ ]:
# Histogram of Pearson r values for all genes.
plt.figure(figsize=(10, 6))
sns.histplot(selected_correlations_all['Pearson Correlation'], bins=50, kde=True)
# Red line at r = 0.41.
plt.axvline(x=0.40, color='red', linestyle='--', label='r = 0.40')
plt.title('Distribution of Pearson Correlation Coefficients')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plot r vs -log10(p)value
import numpy as np
plt.figure(figsize=(10, 6))
plt.scatter(selected_correlations_all['Pearson Correlation'], -np.log10(selected_correlations_all['p_value']), alpha=0.6)
plt.axvline(x=0.40, color='red', linestyle='--', label='r = 0.40')
# horizontal line at p-value = 0.05
plt.axhline(y=-np.log10(0.05), color='blue', linestyle='--', label='p-value = 0.05')
plt.title('Pearson Correlation vs -log10(p-value)')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('-log10(p-value)')
plt.show()

In [ ]:
# try for filtered genes : genes with Chronos_min < -1 (FDR is lower)
selected_correlations_filtered = correlate_selected_genes_with_expression(filtered_df, merged_df, expr_col='MTUS1', genes=[])
print(selected_correlations_filtered.shape)
display(selected_correlations_filtered[(selected_correlations_filtered['p_value']< 0.05) & (selected_correlations_filtered['Pearson Correlation'] > 0) &  (selected_correlations_filtered['min_score'] < -1)].sort_values('Pearson Correlation', ascending=False))
#selected_correlations_filtered[(selected_correlations_filtered['p_value']< 0.05) & (selected_correlations_filtered['Pearson Correlation'] > 0) &  (selected_correlations_filtered['min_score'] < -1) ].sort_values('Pearson Correlation', ascending=False).to_csv("" \
#"correlation_MTUS1_expression_CRISPRGeneEffect_TNBC_TPMLogp1_filtered_genes_min_less_than_-1.csv", index=False)


In [ ]:
# Histogram of Pearson r values for all genes.
plt.figure(figsize=(10, 6))
sns.histplot(selected_correlations_filtered['Pearson Correlation'], bins=50, kde=True)
# Red line at r = 0.41.
plt.axvline(x=0.40, color='red', linestyle='--', label='r = 0.40')
plt.title('Distribution of Pearson Correlation Coefficients')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Plot r vs -log10(p)value
import numpy as np
plt.figure(figsize=(10, 6))
plt.scatter(selected_correlations_filtered['Pearson Correlation'], -np.log10(selected_correlations_filtered['p_value']), alpha=0.6)
plt.axvline(x=0.40, color='red', linestyle='--', label='r = 0.40')
# horizontal line at p-value = 0.05
plt.axhline(y=-np.log10(0.05), color='blue', linestyle='--', label='p-value = 0.05')
plt.title('Pearson Correlation vs -log10(p-value)')
plt.xlabel('Pearson Correlation Coefficient')
plt.ylabel('-log10(p-value)')
plt.show()

In [ ]:
selected_correlations[selected_correlations['Gene'] == 'ACTL6A']

In [ ]:
# try for filtered genes # genes with sd > 0.75 (ceux qui ont le plus de variations  sans filtrer Chronos_min < -1)
selected_correlations = correlate_selected_genes_with_expression(effect_df_filtered, merged_df, expr_col='MTUS1', genes=[])

display(selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0) &  (selected_correlations['min_score'] < -1)].sort_values('Pearson Correlation', ascending=False))

In [ ]:

# try for filtered genes # genes with sd > 0.75 and Chronos_min < -1)
selected_correlations = correlate_selected_genes_with_expression(filtered_df_filter, merged_df, expr_col='MTUS1', genes=[])

display(selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0) &  (selected_correlations['min_score'] < -1)].sort_values('Pearson Correlation', ascending=False))

In [ ]:
print(selected_correlations.shape)
for g in ['BAG2', 'HSPA8', 'IRE1', 'USP49','ATF6','PERK','GRP78','SKP1']:
    display(selected_correlations[selected_correlations['Gene']==g]) 

## C. Essential genes and potential context-specific targets in an ATIP3-low setting.
When ATIP3 is low, Chronos scores are more negative: the gene is more essential in ATIP3-low cells.
These genes become candidate dependencies in the absence or low expression of ATIP3.

ATIP3 expression positively correlated with Chronos score, with Chronos score < 0.
Functional dependency: the gene supports survival and may represent a candidate therapeutic target in an ATIP3-low context.

MYC, DDX10 (ATIP3 interacts with DDX21)

In [ ]:
# try for filtered genes # genes with Chronos_min < -1 (diminue FDR)
selected_correlations = correlate_selected_genes_with_expression(filtered_df, merged_df, expr_col='MTUS1', genes=[])
selected_correlations.head()
#selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0)& (selected_correlations['min_score'] < -1) ].sort_values('Pearson Correlation', ascending=False)

In [ ]:
selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0)& (selected_correlations['min_score'] < -1) ].sort_values('Pearson Correlation', ascending=False).to_csv('essential_genes.csv', sep = ';')

In [ ]:
selected_correlations[(selected_correlations['Gene']=='MYC')] 
 

In [ ]:
gene_list_chronos = selected_correlations[
    (selected_correlations['p_value'] < 0.05) &
    (selected_correlations['Correlation'] > 0) &
    (selected_correlations['min_score'] < -1)
]['Gene'].unique().tolist()
print("Number of genes in the Chronos list:", len(gene_list_chronos))

In [ ]:
gene_list_chronos

In [ ]:
selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Correlation'] > 0) & (selected_correlations['min_score'] < -1)]['Gene'].unique()

In [ ]:
selected_correlations_all[(selected_correlations_all['p_value']< 0.1) & (selected_correlations_all['Pearson Correlation'] < 0) & (selected_correlations_all['min_score'] > 0)]['Gene'].unique()

In [ ]:
def plot_expr_vs_effect( expr_col='MTUS1', effect_col='MYC'):
    plt.figure(figsize=(6, 5))

    sub = pd.merge(
            merged_df[['DepMap_ID', expr_col]],
            effect_df[['DepMap_ID', effect_col]],
            on='DepMap_ID'
        ).dropna()
    
    # Check that there are enough data points.
    if len(sub) < 3:
        print(f"Pas assez de points")
        return

    # Compute correlation.
    r, p = pearsonr(sub[expr_col], sub[effect_col])
    sns.regplot(data=sub, x=expr_col, y=effect_col, scatter_kws={'alpha':0.6})
    #sns.regplot(data=sub[sub['ModelSubtypeFeatures'].str.contains('TNBC', na=False)],x=expr_col, y=effect_col, scatter_kws={'alpha': 0.6}, color='r', label='TNBC')
    #sns.scatterplot(data=sub[sub['ModelSubtypeFeatures'].str.contains('TNBC', na=False)],x=expr_col, y=effect_col, label='TNBC', color='r', alpha=0.6)
    
    

    # Annotation
    plt.title(f"Correlation {expr_col} vs {effect_col}\nr = {r:.2f}, p = {p:.1e}")
    plt.xlabel(f"{expr_col} (log2(TPM+1))")
    plt.ylabel(f"{effect_col}effect (CRISPRGeneEffect)")
    plt.axhline(-1, linestyle='--', color='gray', linewidth=1)
    plt.tight_layout()
    plt.ylim(-3, .5)
    plt.show()

plot_expr_vs_effect('MTUS1','MASTL')#PRMT1. KAT5;  ACTL6A (r = 0.3, p = 0.1) HDAC3 (r = 0.13, p = 0.5);  EEF2(r = 0.4 p=0.07);  MASTL (r = 0.5 p=0.02); DYNLL1 (r = 0.52 p=0.009)

In [ ]:
# don't work with protein quantity

def plot_expr_vs_expression( expr_col='MTUS1', expression_col='MYC'):
    plt.figure(figsize=(6, 5))

    sub = pd.merge(
            merged_df[['DepMap_ID', expr_col]],
            merged_df[['DepMap_ID', expression_col]],
            on='DepMap_ID'
        ).dropna()
    
    # Check that there are enough data points.
    if len(sub) < 3:
        print(f"Pas assez de points")
        return

    # Compute correlation.
    r, p = pearsonr(sub[expr_col], sub[expression_col])
    sns.regplot(data=sub, x=expr_col, y=expression_col, scatter_kws={'alpha':0.6})
    #sns.regplot(data=sub[sub['ModelSubtypeFeatures'].str.contains('TNBC', na=False)],x=expr_col, y=expression_col, scatter_kws={'alpha': 0.6}, color='r', label='TNBC')
    #sns.scatterplot(data=sub[sub['ModelSubtypeFeatures'].str.contains('TNBC', na=False)],x=expr_col, y=expression_col, label='TNBC', color='r', alpha=0.6)
    
    

    # Annotation
    plt.title(f"Correlation {expr_col} vs {expression_col}\nr = {r:.2f}, p = {p:.1e}")
    plt.xlabel(f"{expr_col} (log2(TPM+1))")
    plt.ylabel(f"{expression_col} (log2(TPM+1))")
    plt.axhline(1, linestyle='--', color='gray', linewidth=0.5)
    plt.tight_layout()
    plt.show()

plot_expr_vs_expression('MTUS1','MYC')

## D. ORA with Hallmark

In [ ]:
# default: Human
import gseapy as gp
names = gp.get_library_name()
names

In [ ]:



# Gene list.
gene_list = selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#essential_genes.to_list()
#selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#gene_list_chronos#
print("Number of genes in the list:", len(gene_list))

enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=['MSigDB_Hallmark_2020'],#'Proteomics_Drug_Atlas_2023',    # ou 'Hallmark_2020' 'MSigDB_Hallmark_2020'
    organism='Human',
    outdir=None,
)

enr.res2d[enr.res2d['Adjusted P-value'] < 1][['Term','Adjusted P-value','Overlap','Genes']].head(53)

In [ ]:
enr.res2d[enr.res2d['Term'].str.contains('DNA damage', case=False, na=False)][['Term','Adjusted P-value','Overlap','Genes']]

In [ ]:
# With KEGG
import gseapy as gp

# Gene list.
gene_list = selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#essential_genes.to_list()
#selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#gene_list_chronos#
print("Number of genes in the list:", len(gene_list))

enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=['KEGG_2021_Human'],    # ou 'Hallmark_2020'
    organism='Human',
    outdir=None,
)

t_Kegg = enr.res2d[['Term','Adjusted P-value','Overlap','Genes']]
t_Kegg[t_Kegg['Adjusted P-value'] < 0.3]


In [ ]:
# With GO_Biological_Process_2025
import gseapy as gp

# Gene list.
gene_list = selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Pearson Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#essential_genes.to_list()
#selected_correlations[(selected_correlations['p_value']< 0.05) & (selected_correlations['Correlation'] > 0)& (selected_correlations['min_score'] < -1)  ]['Gene'].unique().tolist()
#gene_list_chronos#
print("Number of genes in the list:", len(gene_list))

enr = gp.enrichr(
    gene_list=gene_list,
    gene_sets=['GO_Biological_Process_2025'],    # ou 'Hallmark_2020'
    organism='Human',
    outdir=None,
)

t_GO = enr.res2d[['Term','Adjusted P-value','Overlap','Genes']]
t_GO[t_GO['Adjusted P-value'] < 0.1]

In [ ]:
import numpy as np
import pandas as pd
import gseapy as gp

# Start from a `selected_correlations` DataFrame with at least:
# ['Gene', 'Correlation', 'p_value', 'min_score']
# -> 'Correlation' = corr(MTUS1, Chronos), with sign preserved.
# -> 'p_value' = correlation p-value.
# -> 'min_score' = minimum Chronos score, used when applying a biological filter.

df = selected_correlations.dropna(subset=['Gene', 'Correlation', 'p_value']).copy()

# OPTION A (simple and standard): use the correlation as the ranking score.
df['score_A'] = df['Correlation']

# OPTION B (more informative): signed score weighted by significance.
# (larger weight when p is small, while keeping the correlation sign).
df['score_B'] = -np.log10(df['p_value'].clip(lower=1e-300)) * np.sign(df['Correlation'])

# Choose the preferred ranking option.
rnk = df[['Gene', 'score_A']].rename(columns={'Gene':'gene', 'score_A':'score'})

# Sort in decreasing order (gseapy.prerank expects ranking from largest to smallest).
rnk = rnk.sort_values('score', ascending=False)

# Optional: restrict to biologically plausible genes.
# Example: apply min_score < -1 before sorting, noting that this introduces a bias.
# rnk = rnk[df['min_score'] < -1].sort_values('score', ascending=False)

# Lancer GSEA prerank (Hallmark)
pre_res = gp.prerank(
    rnk=rnk,  # Two-column dataframe: gene, score.
    gene_sets='MSigDB_Hallmark_2020',  # ou 'Hallmark_2020' selon ta version
    processes=4,
    permutation_num=1000,   # 1000 permutations = standard
    outdir=None,            # No disk output.
    seed=42,
    min_size=15,            # Standard Hallmark bounds
    max_size=500
)

# GSEA results.
res = pre_res.res2d
# Colonnes usuelles: 'es','nes','pval','fdr','lead_genes','geneset_size','matched_size',...
# Afficher les termes les plus significatifs (FDR croissante)
#res_sorted = res.sort_values('fdr')
#res_sorted[['term','nes','pval','fdr','lead_genes']].head(20)
res[res['FDR q-val'] < 0.05].sort_values('NES', ascending=False)

In [ ]:
import pandas as pd

from gseapy import Msigdb
msig = Msigdb()
# Find MSigDB categories.


In [ ]:
import pandas as pd

#from gseapy import Msigdb
#msig = Msigdb()

# Build a dataframe from a gp.get_library(name='MSigDB_Hallmark_2020') dictionary.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='MSigDB_Hallmark_2020'), orient="index") # Previous option; may not be up to date.
#MSigDB = pd.DataFrame.from_dict(gp.get_library(name='WikiPathways_2024_Human'), orient="index") # Alternative option to test.
MSigDB = pd.DataFrame.from_dict(msig.get_gmt(category='h.all', dbver='2024.1.Hs'),orient="index")  # 50 gene sets
WikiPathways_2024_Human = pd.DataFrame.from_dict(gp.get_library(name='WikiPathways_2024_Human'),orient="index")
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = MSigDB.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(MSigDB.iloc[i,j])
        gene_set_names.append(MSigDB.index[i])

# dataframe
MSigDB = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
MSigDB = MSigDB[MSigDB['genesymbol'].isnull() == False]
MSigDB[MSigDB['genesymbol'] == ''] #PSMB5 DDX10 PHF8 (pas included) ACTL6A

In [ ]:
WikiPathways_2024_Human = pd.DataFrame.from_dict(gp.get_library(name='WikiPathways_2024_Human'),orient="index")
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = WikiPathways_2024_Human.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(WikiPathways_2024_Human.iloc[i,j])
        gene_set_names.append(WikiPathways_2024_Human.index[i])

# dataframe
WikiPathways_2024_Human = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
WikiPathways_2024_Human = WikiPathways_2024_Human[WikiPathways_2024_Human['genesymbol'].isnull() == False]
WikiPathways_2024_Human[WikiPathways_2024_Human['genesymbol'] == 'ACTL6A'] #PSMB5 DDX10 PHF8 (pas included)


In [ ]:

GO_Biological_Process_2025 = pd.DataFrame.from_dict(gp.get_library(name='GO_Biological_Process_2025'),orient="index")
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = GO_Biological_Process_2025.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(GO_Biological_Process_2025.iloc[i,j])
        gene_set_names.append(GO_Biological_Process_2025.index[i])

# dataframe
GO_Biological_Process_2025 = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
GO_Biological_Process_2025 = GO_Biological_Process_2025[GO_Biological_Process_2025['genesymbol'].isnull() == False]
GO_Biological_Process_2025[GO_Biological_Process_2025['genesymbol'] == 'DDX10'] #PSMB5 DDX10 PHF8 (pas included)

In [ ]:
KEGG_2021_Human = pd.DataFrame.from_dict(gp.get_library(name='KEGG_2021_Human'),orient="index")
# Reshape the table so that gene symbols are in one column and gene-set names in another column.
# List containing all values.
gene_symbols = []
gene_set_names = []
k,l = KEGG_2021_Human.shape
for i in range(0,k):
    for j in range(0,l):
        gene_symbols.append(KEGG_2021_Human.iloc[i,j])
        gene_set_names.append(KEGG_2021_Human.index[i])

# dataframe
KEGG_2021_Human = pd.DataFrame({'geneset':gene_set_names,'genesymbol':gene_symbols})
# on remove les Noone
KEGG_2021_Human = KEGG_2021_Human[KEGG_2021_Human['genesymbol'].isnull() == False]
KEGG_2021_Human[KEGG_2021_Human['genesymbol'] == 'DDX10'] #PSMB5 DDX10 PHF8 (pas included)


In [ ]:
from IPython.display import display
gen ='EP400'  
display(MSigDB[MSigDB['genesymbol'] == gen])
display(KEGG_2021_Human[KEGG_2021_Human['genesymbol'] == gen])
display(WikiPathways_2024_Human[WikiPathways_2024_Human['genesymbol'] == gen])
display(GO_Biological_Process_2025[GO_Biological_Process_2025['genesymbol'] == gen])

In [ ]:
MSigDB[MSigDB['genesymbol'] == 'MYC'].empty

In [ ]:
# Identify which genes from gene_list_chronos are present in the Hallmark database.
n = 0
n1 = 0
n2 = 0
for g in gene_list_chronos:
    path = MSigDB[MSigDB['genesymbol'] == g]
    n +=1
    if not path.empty:
        display(path)
        n1+=1
    else:
        #print(g, "is not in the Hallmark ddatabase")
        n2+=1
print(f"There are {n} DE genes, {n1} in Halmmark db and {n2} not")
